# AgriSense - Feature Engineering: Price Volatility &amp; Rolling Averages

Welcome to Section 1.1 &amp; 1.2! In this notebook, we move beyond basic analysis and start doing **Feature Engineering**.

### What is Feature Engineering?
Feature Engineering means taking existing raw data (like daily prices) and using math to create entirely **new** columns (features) that reveal hidden patterns. These new features make our Machine Learning models significantly smarter than just feeding them raw data.

### Our Goals:
1. **Price Volatility Index:** Measure how wildly the price swings up and down over 7 and 30 days. High volatility means high risk for farmers.
2. **Rolling Average Price:** Smooth out the daily price jumps to find the clear, underlying trend (is the real price actually going up or down?).

In [ ]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Create necessary directories for saving processed data and figures
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../outputs/figures").mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print("Libraries imported and directories ready!")

## Loading &amp; Preparing Data for Rolling Calculations

To calculate a 30-day rolling average, we need 30 consecutive days of data. Let's generate a continuous 90-day time-series for 3 crops (Wheat, Rice, Tomato) to perfectly demonstrate how rolling windows work.

In [ ]:
# 2. Generate 90 days of continuous price data for 3 crops
np.random.seed(42)

dates = pd.date_range(start="2024-01-01", periods=90, freq='D')
crops = ['Wheat', 'Rice', 'Tomato']

data = []
for crop in crops:
    # Base prices: Wheat = ~2400, Rice = ~3200, Tomato = ~1500 (highly volatile)
    if crop == 'Wheat':
        prices = 2400 + np.sin(np.linspace(0, 5, 90)) * 50 + np.random.normal(0, 15, 90)
    elif crop == 'Rice':
        prices = 3200 + np.cos(np.linspace(0, 5, 90)) * 60 + np.random.normal(0, 20, 90)
    else: # Tomato (high volatility)
        prices = 1500 + np.sin(np.linspace(0, 10, 90)) * 400 + np.random.normal(0, 150, 90)
        
    for i, date in enumerate(dates):
        data.append({'date': date, 'commodity': crop, 'modal_price': round(prices[i])})

df = pd.DataFrame(data)

# Sort by crop and then by date - MANDATORY step before doing rolling calculations!
df = df.sort_values(by=['commodity', 'date']).reset_index(drop=True)

print(f"Data generated! Shape: {df.shape}")
display(df.head(3))

## Section 1.1: Price Volatility Index
Volatility refers to the unpredictable movement of prices. If a price jumps up ₹500 today and drops ₹400 tomorrow, it is highly volatile (Risky!).
We measure this by calculating the **Standard Deviation (`std()`)** of prices over the past 7 and 30 days.

*Code Hint: We use `groupby('commodity')` because Wheat's rolling average should only look at Wheat's past prices, not Tomato's!*

In [ ]:
# 3. Calculate 7-day and 30-day Price Volatility (Standard Deviation)
# min_periods=1 ensures that Day 1 doesn't just output NaN (Not a Number)

df['price_volatility_7d'] = df.groupby('commodity')['modal_price'].transform(
    lambda x: x.rolling(window=7, min_periods=1).std()
)

df['price_volatility_30d'] = df.groupby('commodity')['modal_price'].transform(
    lambda x: x.rolling(window=30, min_periods=1).std()
)

# For the very first day, standard deviation is 0 because there is no past data to compare against
# Let's fill those initial NaNs with 0
df.fillna({'price_volatility_7d': 0, 'price_volatility_30d': 0}, inplace=True)

display(df[df['commodity'] == 'Tomato'].head(10))

### Creating a "Market Risk" Label
Numbers are great for ML Models, but humans on the Market Dashboard need simple labels. Let's create a "Low/Medium/High" risk indicator based on our 7-day volatility.

In [ ]:
# 4. Create a categorical Market Risk label based on standard deviation thresholds
def assign_risk(volatility):
    if volatility &lt; 25:
        return 'Low Risk'
    elif volatility &lt; 100:
        return 'Medium Risk'
    else:
        return 'High Risk'

df['market_risk'] = df['price_volatility_7d'].apply(assign_risk)

# Let's see the risk distribution
print(df.groupby(['commodity', 'market_risk']).size().unstack(fill_value=0))

## Section 1.2: Rolling Average Price
Daily prices are noisy. A 7-day rolling average takes the average of the last 7 days to give us a smooth trend line.

In [ ]:
# 5. Calculate 7-day and 30-day Rolling Average (Mean)

df['price_rolling_avg_7d'] = df.groupby('commodity')['modal_price'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

df['price_rolling_avg_30d'] = df.groupby('commodity')['modal_price'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean()
)

# Let's look at Tomato prices again to see the smoothed averages
display(df[df['commodity'] == 'Tomato'].tail(5))

## Visualization &amp; Insights
Let's see the magic of Feature Engineering visually by plotting the original noisy price against our smooth rolling averages.

In [ ]:
# 6. Plotting Original vs Rolling Averages for TOMATO (our most volatile crop)
tomato_df = df[df['commodity'] == 'Tomato']

plt.figure(figsize=(14, 7))

# Plot the raw, noisy daily price
sns.lineplot(data=tomato_df, x='date', y='modal_price', 
             color='lightgray', marker='o', alpha=0.6, label='Daily Modal Price (Noisy)')

# Plot the 7-day rolling average
sns.lineplot(data=tomato_df, x='date', y='price_rolling_avg_7d', 
             color='orange', linewidth=2, label='7-Day Rolling Average (Smooth)')

# Plot the 30-day rolling average
sns.lineplot(data=tomato_df, x='date', y='price_rolling_avg_30d', 
             color='red', linewidth=3, label='30-Day Rolling Average (Very Smooth)')

plt.title('Feature Engineering: Tomato Price Smoothing', fontsize=18)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (₹)', fontsize=12)
plt.legend(fontsize=12)

plt.savefig('../outputs/figures/rolling_average_tomato.png', dpi=300, bbox_inches='tight')
plt.show()

**💡 Insight:** 
* "Rolling average gives a smoother trend for the Market page." Instead of showing farmers a chaotic zigzag line (gray), we can show them the solid red line so they understand the *actual* season trend!

In [ ]:
# 7. Plotting Volatility (Market Risk) Over Time
plt.figure(figsize=(14, 5))

# Compare volatility of Wheat vs Tomato
sns.lineplot(data=df[df['commodity'].isin(['Wheat', 'Tomato'])], 
             x='date', y='price_volatility_7d', hue='commodity', linewidth=2)

plt.title('Price Volatility (Risk) Over Time: Wheat vs Tomato', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('7-Day Volatility (Std Dev)', fontsize=12)

plt.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='High Risk Threshold')
plt.legend()

plt.savefig('../outputs/figures/volatility_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

**💡 Insight:** 
* "High volatility periods indicate higher market risk for farmers."
* As the chart proves, Wheat stays safely below the High Risk line practically all season.
* Tomato repeatedly crosses the High Risk line, which means we should alert farmers on the frontend UI!

## Save our Enhanced Dataset!
We've added 5 amazing new columns to our data. This completely changes the game for our ML models. Let's save this "Enhanced" dataset for the next step.

In [ ]:
# 8. Save the processed and feature-engineered dataframe
output_path = Path('../data/processed/crop_prices_enhanced.csv')
df.to_csv(output_path, index=False)

print(f"Successfully saved enhanced dataset with {len(df.columns)} columns to {output_path}!")
print("New columns added:")
for col in ['price_volatility_7d', 'price_volatility_30d', 'market_risk', 'price_rolling_avg_7d', 'price_rolling_avg_30d']:
    print(f" - {col}")

### Next Steps
Congratulations! You've just performed Feature Engineering.
1. The **Rolling Averages** will power the beautiful, smooth charts on our Next.js frontend.
2. The **Volatility** and **Risk Labels** will serve as warning badges on the dashboard for farmers.
3. In the next section, we will take this enhanced `.csv` file and finally feed it into a **Machine Learning Algorithm** to predict future prices based on these trends!